# Funciones de Ventana

- ROW_NUMBER, RANK, LAG, LEAD, FIRST_VALUE — syntax's SQLite, PostgreSQL y MySQL

## Introducción

- Las Window Functions (funciones de ventana) permiten realizar cálculos sobre un conjunto de filas relacionadas con la fila actual, sin colapsar el resultado como lo hace GROUP BY. Con la cláusula OVER() defines la "ventana" de filas sobre la que opera cada función. Son esenciales para rankings, análisis de tendencias, comparaciones con filas anteriores o siguientes, y cálculos acumulados. A diferencia de GROUP BY, cada fila conserva su identidad en el resultado.

### Objetivos de Aprendizaje

- Entender la diferencia entre Window Functions y GROUP BY
- Usar ROW_NUMBER, RANK y DENSE_RANK para rankings con y sin empates
- Aplicar LAG y LEAD para acceder a filas anteriores y siguientes
- Usar FIRST_VALUE, LAST_VALUE y NTH_VALUE para obtener valores extremos por partición
- Conocer las diferencias de sintaxis entre SQLite, PostgreSQL y MySQL

### ¿Qué son las Window Functions?

> Una Window Function realiza un cálculo sobre un conjunto de filas definido por OVER(). A diferencia de GROUP BY que colapsa múltiples filas en una, las Window Functions mantienen todas las filas originales y añaden una columna calculada. La "ventana" puede particionarse (PARTITION BY) para reiniciar el cálculo en cada grupo, y ordenarse (ORDER BY) para definir el contexto de cada fila.

In [ ]:
import sqlite3

conn = sqlite3.connect(':memory:')
cursor = conn.cursor()

cursor.execute('''CREATE TABLE empleados (
    id INTEGER PRIMARY KEY, nombre TEXT, departamento TEXT, ventas REAL)''')
cursor.executemany('INSERT INTO empleados VALUES (?,?,?,?)', [
    (1,'Ana','Norte',85000),(2,'Luis','Norte',72000),
    (3,'Sara','Sur',91000),(4,'Pedro','Sur',68000),
    (5,'María','Norte',79000),(6,'Carlos','Sur',88000),
])
conn.commit()

# GROUP BY colapsa: un resultado por departamento
print("── GROUP BY (colapsa filas) ──")
cursor.execute("""
    SELECT departamento, AVG(ventas) AS promedio
    FROM empleados
    GROUP BY departamento
""")
for row in cursor.fetchall():
    print(f"  ${row[0]}: $${row[1]:,.0f}")

# Window Function: mantiene todas las filas + añade promedio
print("\n── OVER() (mantiene filas originales) ──")
cursor.execute("""
    SELECT nombre, departamento, ventas,
           AVG(ventas) OVER(PARTITION BY departamento) AS promedio_depto,
           ROW_NUMBER() OVER(PARTITION BY departamento ORDER BY ventas DESC) AS ranking
    FROM empleados
    ORDER BY departamento, ranking
""")
print(f"{'Nombre':<8} {'Depto':<6} {'Ventas':>8} {'Prom Depto':>10} {'Rank':>5}")
print("-" * 45)
for row in cursor.fetchall():
    print(f"${row[0]:<8} ${row[1]:<6} $${row[2]:>6,.0f} $${row[3]:>8,.0f} ${row[4]:>5}")

### ROW_NUMBER, RANK y DENSE_RANK

> Estas tres funciones asignan posiciones a las filas dentro de una partición ordenada. La diferencia aparece con empates: ROW_NUMBER asigna números únicos sin importar empates (1,2,3,4). RANK salta números tras un empate (1,2,2,4 — no hay 3). DENSE_RANK no salta (1,2,2,3). Elige según si necesitas posiciones únicas, saltos estilo competición, o clasificación compacta.

In [ ]:
import sqlite3

conn = sqlite3.connect(':memory:')
cursor = conn.cursor()

cursor.execute('''CREATE TABLE vendedores (
    id INTEGER PRIMARY KEY, nombre TEXT, region TEXT, monto REAL)''')
cursor.executemany('INSERT INTO vendedores VALUES (?,?,?,?)', [
    (1,'Ana','Norte',95000),(2,'Luis','Norte',82000),
    (3,'Sara','Norte',82000),(4,'Pedro','Norte',75000),
    (5,'María','Sur',91000),(6,'Carlos','Sur',91000),
    (7,'Elena','Sur',78000),
])
conn.commit()

# Comparar ROW_NUMBER, RANK y DENSE_RANK con empates
cursor.execute("""
    SELECT
        nombre, region, monto,
        ROW_NUMBER()  OVER(PARTITION BY region ORDER BY monto DESC) AS row_num,
        RANK()        OVER(PARTITION BY region ORDER BY monto DESC) AS rank_val,
        DENSE_RANK()  OVER(PARTITION BY region ORDER BY monto DESC) AS dense_rank_val
    FROM vendedores
    ORDER BY region, monto DESC
""")
print(f"{'Nombre':<8} {'Región':<6} {'Monto':>8} {'ROW_NUM':>8} {'RANK':>6} {'DENSE':>6}")
print("-" * 50)
for row in cursor.fetchall():
    print(f"${row[0]:<8} ${row[1]:<6} $${row[2]:>6,.0f} ${row[3]:>8} ${row[4]:>6} ${row[5]:>6}")

# Nota con empates de 82000 en Norte: RANK salta el 3, DENSE_RANK no
print("\n→ Norte 82000 (empate): RANK=2,2 luego salta a 4; DENSE_RANK=2,2 luego 3")

### LAG y LEAD

> LAG accede al valor de una fila anterior dentro de la partición, LEAD accede a la fila siguiente. Son ideales para calcular diferencias entre períodos: variación mes a mes, crecimiento acumulado, o comparar un valor con el del período anterior. Ambas aceptan un offset (cuántas filas saltar, por defecto 1) y un valor por defecto cuando no existe la fila adyacente.

In [ ]:
import sqlite3

conn = sqlite3.connect(':memory:')
cursor = conn.cursor()

cursor.execute('''CREATE TABLE ventas_mes (
    id INTEGER PRIMARY KEY, mes TEXT, region TEXT, monto REAL)''')
cursor.executemany('INSERT INTO ventas_mes VALUES (?,?,?,?)', [
    (1,'2024-01','Norte',45000),(2,'2024-02','Norte',52000),
    (3,'2024-03','Norte',48000),(4,'2024-04','Norte',61000),
    (5,'2024-05','Norte',58000),(6,'2024-06','Norte',67000),
    (7,'2024-01','Sur',38000),(8,'2024-02','Sur',41000),
    (9,'2024-03','Sur',44000),(10,'2024-04','Sur',39000),
])
conn.commit()

# LAG para calcular cambio mes a mes
cursor.execute("""
    SELECT
        mes, region, monto,
        LAG(monto) OVER(PARTITION BY region ORDER BY mes) AS mes_anterior,
        ROUND(
            (monto - LAG(monto) OVER(PARTITION BY region ORDER BY mes))
            * 100.0
            / LAG(monto) OVER(PARTITION BY region ORDER BY mes),
        1) AS cambio_pct
    FROM ventas_mes
    ORDER BY region, mes
""")
print(f"{'Mes':<8} {'Región':<6} {'Monto':>8} {'Anterior':>10} {'Cambio%':>8}")
print("-" * 46)
for row in cursor.fetchall():
    cambio = f"${row[4]:+.1f}%" if row[4] is not None else "  N/A"
    trend = "📈" if (row[4] or 0) > 0 else ("📉" if (row[4] or 0) < 0 else "➡️")
    print(f"${row[0]:<8} ${row[1]:<6} $${row[2]:>6,.0f} {str(row[3] or 'N/A'):>10} {cambio:>8} {trend}")

### FIRST_VALUE, LAST_VALUE y NTH_VALUE


> Estas funciones obtienen el primero, último o enésimo valor dentro de la ventana ordenada. FIRST_VALUE es directo. LAST_VALUE requiere atención: por defecto la ventana va desde el inicio hasta la fila actual (RANGE BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW), por lo que devuelve la fila actual. Para obtener el verdadero último debes especificar ROWS BETWEEN UNBOUNDED PRECEDING AND UNBOUNDED FOLLOWING.

In [ ]:
import sqlite3

conn = sqlite3.connect(':memory:')
cursor = conn.cursor()

cursor.execute('''CREATE TABLE ventas_region (
    id INTEGER PRIMARY KEY, vendedor TEXT, region TEXT, monto REAL)''')
cursor.executemany('INSERT INTO ventas_region VALUES (?,?,?,?)', [
    (1,'Ana','Norte',95000),(2,'Luis','Norte',72000),
    (3,'Sara','Norte',88000),(4,'Pedro','Sur',91000),
    (5,'María','Sur',68000),(6,'Carlos','Sur',85000),
    (7,'Elena','Este',79000),(8,'Diego','Este',84000),
])
conn.commit()

# FIRST_VALUE, LAST_VALUE y NTH_VALUE por región
cursor.execute("""
    SELECT
        vendedor, region, monto,
        FIRST_VALUE(monto)  OVER w AS mejor_venta,
        LAST_VALUE(monto)   OVER w AS peor_venta,
        NTH_VALUE(monto, 2) OVER w AS segunda_venta,
        FIRST_VALUE(vendedor) OVER w AS lider_region
    FROM ventas_region
    WINDOW w AS (
        PARTITION BY region
        ORDER BY monto DESC
        ROWS BETWEEN UNBOUNDED PRECEDING AND UNBOUNDED FOLLOWING
    )
    ORDER BY region, monto DESC
""")
print(f"{'Vendedor':<8} {'Región':<6} {'Monto':>8} {'Mejor':>8} {'Peor':>8} {'2do':>8} {'Líder':<8}")
print("-" * 60)
for row in cursor.fetchall():
    segundo = f"$${row[5]:,.0f}" if row[5] else "  N/A"
    print(f"${row[0]:<8} ${row[1]:<6} $${row[2]:>6,.0f} $${row[3]:>6,.0f} $${row[4]:>6,.0f} {segundo:>8} ${row[6]:<8}")

### Diferencias SQLite / PostgreSQL / MySQL

> Las Window Functions son estándar SQL:2003, pero la compatibilidad varía: SQLite las soporta desde la versión 3.25.0 (2018). PostgreSQL tiene soporte completo incluyendo WINDOW clause y todos los frames. MySQL las añadió en la versión 8.0. La sintaxis básica es idéntica; las diferencias están en características avanzadas como GROUPS frame mode y algunas funciones específicas.

In [ ]:
import sqlite3

conn = sqlite3.connect(':memory:')
cursor = conn.cursor()

cursor.execute('''CREATE TABLE ventas (
    id INTEGER PRIMARY KEY, vendedor TEXT, monto REAL, fecha TEXT)''')
cursor.executemany('INSERT INTO ventas VALUES (?,?,?,?)', [
    (1,'Ana',5000,'2024-01-10'),(2,'Luis',7000,'2024-01-15'),
    (3,'Sara',6500,'2024-02-05'),(4,'Ana',8000,'2024-02-20'),
])
conn.commit()

# Sintaxis compatible SQLite 3.25+ / PostgreSQL / MySQL 8.0
cursor.execute("""
    SELECT vendedor, monto, fecha,
        ROW_NUMBER() OVER(ORDER BY monto DESC) AS ranking,
        SUM(monto) OVER(ORDER BY fecha ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) AS acumulado
    FROM ventas
    ORDER BY ranking
""")
print("SQLite (3.25+) / PostgreSQL / MySQL 8.0 — sintaxis idéntica:")
for row in cursor.fetchall():
    print(f"  ${row[0]:<6} $${row[1]:>5,.0f}  ranking=${row[3]}  acum=$${row[4]:,.0f}")

# Versión SQLite
cursor.execute("SELECT sqlite_version()")
version = cursor.fetchone()[0]
print(f"\nVersión SQLite actual: {version}")

# Notas de compatibilidad (como comentarios de Python):
# PostgreSQL: soporta WINDOW alias, GROUPS frame, FILTER clause
#   SELECT SUM(monto) FILTER (WHERE monto > 5000) OVER() FROM ventas;
# MySQL 8.0+: sintaxis idéntica al estándar, sin WINDOW alias separado
#   SET SESSION sql_mode='ANSI'; -- recomendado para compatibilidad
# SQLite <3.25: NO soporta window functions — actualizar o usar subqueries

## Ejemplos Prácticos

### Ranking de ventas por región

Ejemplo completo en SQLite: crea una tabla con 15+ filas, aplica ROW_NUMBER() OVER(PARTITION BY region ORDER BY monto DESC) y usa un CTE para mostrar el top-3 por región.

In [ ]:
import sqlite3

conn = sqlite3.connect(':memory:')
cursor = conn.cursor()

cursor.execute('''CREATE TABLE ventas_regionales (
    id INTEGER PRIMARY KEY, vendedor TEXT, region TEXT,
    categoria TEXT, monto REAL)''')
cursor.executemany('INSERT INTO ventas_regionales VALUES (?,?,?,?,?)', [
    (1,'Ana Martínez','Norte','Software',95000),
    (2,'Luis García','Norte','Hardware',82000),
    (3,'Sara López','Norte','Servicios',88000),
    (4,'Pedro Ruiz','Norte','Software',76000),
    (5,'Carmen Díaz','Norte','Hardware',91000),
    (6,'Miguel Torres','Sur','Software',87000),
    (7,'Elena Sánchez','Sur','Hardware',93000),
    (8,'Roberto Vega','Sur','Servicios',71000),
    (9,'Isabel Castro','Sur','Software',89000),
    (10,'Fernando Gil','Sur','Hardware',84000),
    (11,'Laura Moreno','Este','Software',78000),
    (12,'Javier Herrera','Este','Hardware',92000),
    (13,'Patricia Jiménez','Este','Servicios',85000),
    (14,'Andrés Navarro','Este','Software',81000),
    (15,'Cristina Flores','Este','Hardware',74000),
    (16,'Raúl Romero','Oeste','Servicios',96000),
    (17,'Marta Blanco','Oeste','Software',88000),
    (18,'David Serrano','Oeste','Hardware',79000),
])
conn.commit()

# CTE con ranking por región
cursor.execute("""
    WITH ranking_regional AS (
        SELECT
            vendedor, region, categoria, monto,
            ROW_NUMBER() OVER(
                PARTITION BY region
                ORDER BY monto DESC
            ) AS rank_region,
            SUM(monto) OVER(PARTITION BY region) AS total_region,
            ROUND(monto * 100.0 / SUM(monto) OVER(PARTITION BY region), 1) AS pct_region
        FROM ventas_regionales
    )
    SELECT vendedor, region, categoria, monto, rank_region, pct_region, total_region
    FROM ranking_regional
    WHERE rank_region <= 3
    ORDER BY region, rank_region
""")

region_actual = None
for row in cursor.fetchall():
    if row[1] != region_actual:
        region_actual = row[1]
        print(f"\n📍 Región ${row[1]} (Total: $${row[6]:,.0f})")
        print(f"  {'#':<3} {'Vendedor':<20} {'Categoría':<10} {'Monto':>8} {'% Región':>9}")
        print("  " + "-" * 55)
    medalla = ['🥇','🥈','🥉'][row[4]-1]
    print(f"  {medalla} ${row[0]:<20} ${row[2]:<10} $${row[3]:>6,.0f} ${row[5]:>8.1f}%")

### Análisis de tendencia mes a mes

Usa LAG() para calcular el crecimiento porcentual mes a mes y detectar los meses donde las ventas cayeron (crecimiento negativo).

In [ ]:
import sqlite3

conn = sqlite3.connect(':memory:')
cursor = conn.cursor()

cursor.execute('''CREATE TABLE historial_ventas (
    id INTEGER PRIMARY KEY, mes TEXT, region TEXT, monto REAL)''')
cursor.executemany('INSERT INTO historial_ventas VALUES (?,?,?,?)', [
    (1,'2024-01','Norte',42000),(2,'2024-02','Norte',45500),
    (3,'2024-03','Norte',41000),(4,'2024-04','Norte',53000),
    (5,'2024-05','Norte',58000),(6,'2024-06','Norte',55000),
    (7,'2024-07','Norte',62000),(8,'2024-08','Norte',59000),
    (9,'2024-09','Norte',67000),(10,'2024-10','Norte',71000),
    (11,'2024-11','Norte',68000),(12,'2024-12','Norte',74000),
    (13,'2024-01','Sur',35000),(14,'2024-02','Sur',38000),
    (15,'2024-03','Sur',36500),(16,'2024-04','Sur',41000),
    (17,'2024-05','Sur',44000),(18,'2024-06','Sur',42500),
])
conn.commit()

# Análisis MoM con LAG y clasificación de tendencia
cursor.execute("""
    WITH tendencia AS (
        SELECT
            mes, region, monto,
            LAG(monto) OVER(PARTITION BY region ORDER BY mes) AS monto_anterior,
            monto - LAG(monto) OVER(PARTITION BY region ORDER BY mes) AS variacion,
            ROUND(
                (monto - LAG(monto) OVER(PARTITION BY region ORDER BY mes))
                * 100.0
                / LAG(monto) OVER(PARTITION BY region ORDER BY mes),
            1) AS crecimiento_pct
        FROM historial_ventas
    )
    SELECT mes, region, monto, monto_anterior, variacion, crecimiento_pct,
           CASE
               WHEN crecimiento_pct IS NULL THEN 'Inicial'
               WHEN crecimiento_pct >= 5    THEN 'Fuerte alza 🚀'
               WHEN crecimiento_pct > 0     THEN 'Alza leve 📈'
               WHEN crecimiento_pct = 0     THEN 'Estable ➡️'
               WHEN crecimiento_pct >= -5   THEN 'Baja leve 📉'
               ELSE                              'Caída fuerte ⚠️'
           END AS clasificacion
    FROM tendencia
    ORDER BY region, mes
""")

print(f"{'Mes':<8} {'Región':<6} {'Monto':>8} {'Anterior':>9} {'Var':>7} {'%':>6}  Tendencia")
print("-" * 65)
region_actual = None
for row in cursor.fetchall():
    if row[1] != region_actual:
        if region_actual:
            print()
        region_actual = row[1]
    var_str = f"+$${row[4]:,.0f}" if (row[4] or 0) > 0 else (f"-${abs(row[4]):,.0f}" if row[4] else "  N/A")
    pct_str = f"${row[5]:+.1f}%" if row[5] is not None else "  N/A"
    print(f"${row[0]:<8} ${row[1]:<6} $${row[2]:>6,.0f} {str(row[3] or 'N/A'):>9} {var_str:>8} {pct_str:>6}  ${row[6]}")

# Resumen: meses con caída
print("\n── Meses con caída de ventas ──")
cursor.execute("""
    WITH tendencia AS (
        SELECT mes, region, monto,
            ROUND(
                (monto - LAG(monto) OVER(PARTITION BY region ORDER BY mes))
                * 100.0
                / LAG(monto) OVER(PARTITION BY region ORDER BY mes),
            1) AS crecimiento_pct
        FROM historial_ventas
    )
    SELECT mes, region, monto, crecimiento_pct
    FROM tendencia
    WHERE crecimiento_pct < 0
    ORDER BY crecimiento_pct
""")
for row in cursor.fetchall():
    print(f"  ⚠️  ${row[0]} | ${row[1]:<6} | $${row[2]:,.0f} | ${row[3]:+.1f}%")

## Tips y Mejores Prácticas

> SQLite soporta Window Functions solo desde la versión 3.25.0 (septiembre 2018). Verifica tu versión con: SELECT sqlite_version(). Si usas una versión anterior, deberás actualizar o reescribir las queries con subqueries correlacionadas.

> ROWS vs RANGE en el frame de la ventana: ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW considera filas físicas (más predecible). RANGE BETWEEN hace lo mismo pero agrupa filas con el mismo valor de ORDER BY. Para LAST_VALUE correcto, siempre especifica ROWS BETWEEN UNBOUNDED PRECEDING AND UNBOUNDED FOLLOWING.

> Evita aplicar Window Functions sobre tablas grandes sin índices en las columnas de PARTITION BY y ORDER BY. Cada función de ventana requiere ordenar la partición, lo que puede ser costoso. Crea índices en las columnas más usadas en OVER().

> Usa CTEs para hacer legibles las queries con Window Functions: calcula los valores de ventana en el primer CTE, luego filtra y presenta en el SELECT final. Así evitas el error común de usar WHERE sobre resultados de funciones de ventana (que debe hacerse en un CTE o subquery exterior).

## Errores Comunes

### Usar WHERE para filtrar resultados de Window Functions directamente
¿Por qué ocurre?
- Las Window Functions se calculan DESPUÉS del WHERE, por lo que no puedes filtrar por el resultado de ROW_NUMBER() o RANK() en la misma query con WHERE rank = 1.

Solución
- Envuelve la query en un CTE o subquery y filtra en el exterior: WITH cte AS (SELECT ..., ROW_NUMBER() OVER(...) AS rn FROM ...) SELECT * FROM cte WHERE rn = 1.

### Confundir PARTITION BY con GROUP BY
¿Por qué ocurre?
- GROUP BY colapsa las filas de cada grupo en una sola. PARTITION BY en una Window Function divide las filas en grupos para el cálculo pero mantiene TODAS las filas en el resultado. Son herramientas diferentes para propósitos distintos.

Solución
- Usa GROUP BY cuando necesitas un resultado agregado por grupo (una fila por grupo). Usa PARTITION BY en OVER() cuando necesitas un valor calculado por grupo pero conservando todas las filas originales.

### LAST_VALUE devuelve el valor actual en lugar del último de la partición
¿Por qué ocurre?
- El frame por defecto de una Window Function es RANGE BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW. Así, LAST_VALUE ve solo hasta la fila actual, por lo que devuelve el valor de esa misma fila.

Solución
- Especifica explícitamente el frame completo: LAST_VALUE(col) OVER(PARTITION BY ... ORDER BY ... ROWS BETWEEN UNBOUNDED PRECEDING AND UNBOUNDED FOLLOWING).